# FINN notebook para la red propia de Miguel

Este notebook está preparado para usar los pesos `state_dict_quantized_last.pth` y generar el modelo `model.onnx` para FINN.

La red incluida es la arquitectura `BobEsponja`, con:

- entrada: imagen en escala de grises `1 x 64 x 64`
- salida: `14` clases
- activaciones cuantizadas a `4` bits
- entrada cuantizada a `8` bits
- pesos cargados desde `state_dict_quantized_last.pth`

> Importante: el archivo `.pth` no se abre ni se descomprime. PyTorch lo carga directamente con `torch.load(...)`.

## 1. Preparación del entorno

Esta celda configura rutas y variables de entorno necesarias para FINN. Si `FINN_BUILD_DIR` no está definido, FINN falla al generar carpetas temporales.

In [6]:
import os
import sys
import gc
import warnings
import shutil
from pathlib import Path

warnings.filterwarnings("default")
gc.collect()

# Carpeta de trabajo del notebook
print("Carpeta actual:", os.getcwd())

# FINN necesita esta variable para crear directorios temporales de construcción
os.environ["FINN_BUILD_DIR"] = os.path.abspath("FINN_build")
os.makedirs(os.environ["FINN_BUILD_DIR"], exist_ok=True)
print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])

# Comprobación rápida de que estamos dentro del entorno FINN
print("Python executable:", sys.executable)
print("Python path[0]:", sys.path[0])

Carpeta actual: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example
FINN_BUILD_DIR = /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build
Python executable: /opt/conda/bin/python
Python path[0]: /home/miguel_tfg/FINN-v0.9-Stable/src


## 2. Importación de librerías

Aquí se cargan PyTorch, Brevitas y las utilidades de FINN/QONNX.

In [7]:
import torch
import torch.nn as nn
import brevitas.nn as qnn
import brevitas.onnx as bo
from brevitas.quant import Int8Bias

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup

from finn.util.visualization import showInNetron
from finn.builder import build_dataflow as build

print("Torch cargado. CUDA disponible:", torch.cuda.is_available())

Torch cargado. CUDA disponible: False


## 3. Definición de la red

Esta arquitectura debe coincidir exactamente con la arquitectura usada durante el entrenamiento del archivo `state_dict_quantized_last.pth`.

Si aparece un error de `Missing key`, `Unexpected key` o `size mismatch`, significa que los pesos no corresponden a esta arquitectura y habría que cambiar esta clase por la clase real de entrenamiento.

In [12]:
class RedMiguel(nn.Module):
    def __init__(self, in_width=8, weight_width=4, act_width=4, out_width=8):
        super(RedMiguel, self).__init__()

        self.quant_input = qnn.QuantIdentity(
            return_quant_tensor=True,
            bit_width=in_width
        )

        self.fc1 = qnn.QuantLinear(
            in_features=576,
            out_features=256,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu1 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.fc2 = qnn.QuantLinear(
            in_features=256,
            out_features=128,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu2 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.fx = qnn.QuantLinear(
            in_features=128,
            out_features=16,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu3 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.predictor = qnn.QuantLinear(
            in_features=16,
            out_features=2,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.quant_output = qnn.QuantIdentity(
            return_quant_tensor=True,
            bit_width=out_width
        )

    def forward(self, x):
        x = self.quant_input(x)

        # La red espera 576 valores: 16 x 12 x 3
        x = x.reshape(x.size(0), -1)

        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fx(x)
        x = self.relu3(x)

        x = self.predictor(x)
        x = self.quant_output(x)

        return x

## 4. Carga de los pesos propios

Esta celda carga tu archivo `state_dict_quantized_last.pth`. Debe estar en la misma carpeta que este notebook.

In [13]:
WEIGHTS_FILE = "state_dict_quantized_last.pth"

model = RedMiguel(in_width=8, weight_width=4, act_width=4, out_width=8)

state_dict = torch.load(WEIGHTS_FILE, map_location="cpu")

if isinstance(state_dict, dict) and "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

model.load_state_dict(state_dict)
model.eval()

print("Pesos cargados correctamente desde:", WEIGHTS_FILE)

Pesos cargados correctamente desde: state_dict_quantized_last.pth


In [14]:
import torch

state_dict = torch.load("state_dict_quantized_last.pth", map_location="cpu")

if isinstance(state_dict, dict) and "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

for k, v in state_dict.items():
    if hasattr(v, "shape"):
        print(k, v.shape)
    else:
        print(k, type(v))

quant_input.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fc1.weight torch.Size([256, 576])
fc1.bias torch.Size([256])
relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fc2.weight torch.Size([128, 256])
fc2.bias torch.Size([128])
relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fx.weight torch.Size([16, 128])
fx.bias torch.Size([16])
relu3.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
predictor.weight torch.Size([2, 16])
predictor.bias torch.Size([2])
quant_output.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])


## 5. Prueba rápida de inferencia en PyTorch

Esto no valida la precisión de la red, pero comprueba que la arquitectura acepta una entrada `1x1x64x64` y devuelve una salida de `14` valores.

In [15]:
with torch.no_grad():
    dummy_input = torch.randn(*INPUT_SHAPE)
    dummy_output = model(dummy_input)

print("Forma entrada:", tuple(dummy_input.shape))
print("Forma salida:", tuple(dummy_output.shape))
print("Salida ejemplo:", dummy_output)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x4096 and 576x256)

## 6. Exportación a ONNX compatible con FINN

Aquí se crea `model.onnx`, que será la red que FINN transforma internamente.

In [ ]:
model_file_name = "model.onnx"

bo.export_finn_onnx(
    model,
    (1, 3, 16, 12),
    model_file_name
)

print("Modelo exportado a:", model_file_name)

## 7. Comprobación de entrada, salida y capas del ONNX

Esta celda sirve para verificar que FINN está viendo la red correcta.

In [ ]:
model_check = ModelWrapper(MODEL_ONNX)

print("Entradas:")
for inp in model_check.graph.input:
    print("  ", inp.name, model_check.get_tensor_shape(inp.name))

print("\nSalidas:")
for out in model_check.graph.output:
    print("  ", out.name, model_check.get_tensor_shape(out.name))

print("\nPrimeros nodos del grafo:")
for node in model_check.graph.node[:30]:
    print("  ", node.op_type, node.name)

## 8. Visualización opcional con Netron

Si Netron funciona en tu entorno, esta celda abre una visualización del modelo. Si da problemas, se puede omitir.

In [ ]:
# Descomenta esta línea si quieres visualizar el modelo:
# showInNetron(MODEL_ONNX)

## 9. Configuración de build_dataflow

Esta configuración prepara la generación Dataflow de FINN. Si quieres una construcción completa de hardware, el notebook intentará generar los artefactos disponibles según tu instalación de FINN.

La frecuencia objetivo es 100 MHz, equivalente a un periodo de reloj de 10 ns.

In [ ]:
FINAL_OUTPUT_DIR = "output_final_miguel"
FOLDING_CONFIG = "folding_config_3cnv.json"

# Limpia resultados anteriores para evitar mezclar ejecuciones
if os.path.exists(FINAL_OUTPUT_DIR):
    shutil.rmtree(FINAL_OUTPUT_DIR)
    print("Resultados anteriores borrados:", FINAL_OUTPUT_DIR)

assert os.path.isfile(FOLDING_CONFIG), f"No se encuentra {FOLDING_CONFIG}."

cfg = build.DataflowBuildConfig(
    output_dir=FINAL_OUTPUT_DIR,
    folding_config_file=FOLDING_CONFIG,
    auto_fifo_depths=True,
    mvau_wwidth_max=36,
    stitched_ip_gen_dcp=True,
    split_large_fifos=True,
    synth_clk_period_ns=10.0,
    target_fps=100,
)

print("Configuración creada.")
print("Output dir:", FINAL_OUTPUT_DIR)
print("Folding:", FOLDING_CONFIG)
print("Periodo reloj síntesis: 10 ns -> 100 MHz")

## 10. Ejecución del build de FINN

Esta es la celda pesada. Puede tardar bastante. Si solo se generan `intermediate_models` y `report`, significa que tu configuración/versión de FINN ha ejecutado solo una parte del flujo o que faltan etapas de generación de IP.

In [ ]:
# Ejecuta el flujo de construcción de FINN
build.build_dataflow_cfg(MODEL_ONNX, cfg)
print("Build dataflow finalizado.")

## 11. Buscar resultados generados

Esta celda busca los archivos más importantes: Verilog, wrapper, DCP, informes y modelos intermedios.

In [ ]:
print("\nCarpetas principales en", FINAL_OUTPUT_DIR)
if os.path.exists(FINAL_OUTPUT_DIR):
    for item in os.listdir(FINAL_OUTPUT_DIR):
        print("  ", item)
else:
    print("No existe", FINAL_OUTPUT_DIR)

print("\nBúsqueda de archivos importantes:")
patterns = ("all_verilog_srcs", "finn_design_wrapper", ".v", ".sv", ".dcp", ".json", ".onnx")
count = 0
for root, dirs, files in os.walk(FINAL_OUTPUT_DIR):
    for f in files:
        if ("all_verilog_srcs" in f or "finn_design_wrapper" in f or f.endswith((".v", ".sv", ".dcp"))):
            print(os.path.join(root, f))
            count += 1

if count == 0:
    print("No se han encontrado todavía archivos Verilog/DCP en output_final_miguel.")
    print("Revisa si el build ha terminado todas las etapas o si faltan opciones de generación de IP.")

## 12. Resumen para documentar en el TFG

Al ejecutar este notebook, el flujo realizado es:

`state_dict_quantized_last.pth` → carga de pesos PyTorch → exportación `model.onnx` → transformaciones FINN → carpeta `output_final_miguel`.

Si se genera la parte hardware, los archivos de interés para Verilator/Vivado serán especialmente:

- `all_verilog_srcs.txt`
- `finn_design_wrapper.v`
- carpetas de IP o proyecto Vivado
- informes de recursos y rendimiento